In [1]:
import os
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import time
import warnings
warnings.filterwarnings('ignore')

os.chdir(r'C:\Users\samru\OneDrive\Desktop\HerCycleAI')

print("Current folder:", os.getcwd())
print("Libraries loaded ✓")

Current folder: C:\Users\samru\OneDrive\Desktop\HerCycleAI
Libraries loaded ✓


In [2]:
from sklearn.metrics import (accuracy_score, precision_score, 
                              recall_score, f1_score, confusion_matrix)

# Load model
print("Loading model...")
model = models.googlenet(pretrained=True)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Load ImageNet labels
with open("imagenet_classes.txt") as f:
    labels = [line.strip() for line in f.readlines()]

print(f"Model ready on {device} ✓")
print(f"Labels loaded: {len(labels)} classes ✓")

Loading model...
Model ready on cpu ✓
Labels loaded: 1000 classes ✓


In [3]:
PCOD_FRIENDLY_KEYWORDS = [
    # eggs - model sees corn/cauliflower for yellow eggs
    'egg', 'hen', 'cock', 'chicken', 'omelette',
    'corn', 'cauliflower', 'ear',
    # fish / salmon
    'salmon', 'fish', 'trout', 'tench', 'coho', 'plate',
    # salad / vegetables
    'salad', 'broccoli', 'spinach', 'cucumber', 'lettuce',
    'vegetable', 'zucchini', 'cabbage', 'kale',
    'tomato', 'pepper', 'mushroom', 'artichoke', 'asparagus',
    # fruits / berries
    'berry', 'strawberry', 'blueberry', 'raspberry', 'fruit',
    'lemon', 'orange', 'apple', 'pomegranate',
    'custard apple', 'granny smith', 'buckeye',
    # legumes
    'lentil', 'bean', 'pea', 'edamame', 'chickpea',
    'soup', 'hot pot', 'wok', 'guacamole', 'pot',
    # nuts / healthy fats
    'almond', 'walnut', 'nut', 'avocado', 'olive',
    # other healthy
    'shrimp', 'crab', 'lobster', 'sashimi', 'miso'
]

PCOD_UNFRIENDLY_KEYWORDS = [
    'donut', 'doughnut', 'pizza', 'french fries', 'ice cream',
    'cake', 'waffle', 'fried', 'bread', 'trifle', 'chocolate',
    'candy', 'pretzel', 'bagel', 'bun', 'pastry', 'cream', 'sundae'
]

def get_pcod_label(imagenet_prediction):
    prediction_lower = imagenet_prediction.lower()
    for keyword in PCOD_FRIENDLY_KEYWORDS:
        if keyword in prediction_lower:
            return "PCOD-Friendly", 1
    for keyword in PCOD_UNFRIENDLY_KEYWORDS:
        if keyword in prediction_lower:
            return "PCOD-Unfriendly", 0
    return "PCOD-Unfriendly", 0

def preprocess_image(image_path):
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    img = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0)
    return tensor.to(device)

print("Decision logic ready ✓")
print("Preprocessing pipeline ready ✓")

Decision logic ready ✓
Preprocessing pipeline ready ✓


In [4]:
# Ground truth for all 10 images
# 1 = Friendly, 0 = Unfriendly
ground_truth = {
    'eggs.jpg'       : 1,
    'greek_salad.jpg': 1,
    'salmon.jpg'     : 1,
    'berries.jpg'    : 1,
    'lentils.jpg'    : 1,
    'donut.jpg'      : 0,
    'pizza.jpg'      : 0,
    'fries.jpg'      : 0,
    'cake.jpg'       : 0,
    'ice_cream.jpg'  : 0
}

PCOD_FRIENDLY_KEYWORDS = [
    'egg', 'hen', 'cock', 'chicken', 'omelette',
    'corn', 'cauliflower', 'ear',
    'salmon', 'fish', 'trout', 'tench', 'coho', 'plate',
    'salad', 'broccoli', 'spinach', 'cucumber', 'lettuce',
    'vegetable', 'zucchini', 'cabbage', 'kale',
    'tomato', 'pepper', 'mushroom', 'artichoke', 'asparagus',
    'berry', 'strawberry', 'blueberry', 'raspberry', 'fruit',
    'lemon', 'orange', 'apple', 'pomegranate',
    'custard apple', 'granny smith', 'buckeye',
    'lentil', 'bean', 'pea', 'edamame', 'chickpea',
    'soup', 'hot pot', 'wok', 'guacamole', 'pot',
    'almond', 'walnut', 'nut', 'avocado', 'olive',
    'shrimp', 'crab', 'lobster', 'sashimi', 'miso'
]

PCOD_UNFRIENDLY_KEYWORDS = [
    'donut', 'doughnut', 'pizza', 'french fries', 'ice cream',
    'cake', 'waffle', 'fried', 'bread', 'trifle', 'chocolate',
    'candy', 'pretzel', 'bagel', 'bun', 'pastry', 'cream', 'sundae'
]

def preprocess_image(image_path):
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    img = Image.open(image_path).convert('RGB')
    return transform(img).unsqueeze(0).to(device)

def get_pcod_label(imagenet_prediction):
    prediction_lower = imagenet_prediction.lower()
    for keyword in PCOD_FRIENDLY_KEYWORDS:
        if keyword in prediction_lower:
            return 1  # Friendly
    for keyword in PCOD_UNFRIENDLY_KEYWORDS:
        if keyword in prediction_lower:
            return 0  # Unfriendly
    return 0  # Default unfriendly if uncertain

# Run inference on all images
y_true = []
y_pred = []
inference_times = []

print("Running inference for metrics...")
print("-" * 40)

for image_file, true_label in ground_truth.items():
    image_path = os.path.join('test_images', image_file)
    
    if not os.path.exists(image_path):
        print(f"  ⚠️  {image_file} not found — skipping")
        continue
    
    tensor = preprocess_image(image_path)
    
    start = time.time()
    with torch.no_grad():
        output = torch.nn.functional.softmax(model(tensor), dim=1)
    elapsed_ms = (time.time() - start) * 1000
    
    confidence, idx = torch.max(output, 1)
    imagenet_label = labels[idx.item()]
    predicted_label = get_pcod_label(imagenet_label)
    
    y_true.append(true_label)
    y_pred.append(predicted_label)
    inference_times.append(elapsed_ms)
    
    correct = "✓" if predicted_label == true_label else "✗"
    label_str = "Friendly" if predicted_label == 1 else "Unfriendly"
    print(f"  {correct} {image_file:<20} → {label_str:<12} ({confidence.item()*100:.1f}%)")

print("-" * 40)
print(f"Inference complete on {len(y_true)} images ✓")

Running inference for metrics...
----------------------------------------
  ✓ eggs.jpg             → Friendly     (49.1%)
  ✓ greek_salad.jpg      → Friendly     (17.2%)
  ✗ salmon.jpg           → Unfriendly   (31.1%)
  ✗ berries.jpg          → Unfriendly   (19.9%)
  ✓ lentils.jpg          → Friendly     (35.3%)
  ✓ donut.jpg            → Unfriendly   (83.0%)
  ✓ pizza.jpg            → Unfriendly   (93.5%)
  ✓ fries.jpg            → Unfriendly   (16.4%)
  ✓ cake.jpg             → Unfriendly   (48.6%)
  ✗ ice_cream.jpg        → Friendly     (42.9%)
----------------------------------------
Inference complete on 10 images ✓


In [5]:
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, confusion_matrix)

# Calculate metrics
accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall    = recall_score(y_true, y_pred)
f1        = f1_score(y_true, y_pred)
cm        = confusion_matrix(y_true, y_pred)
avg_speed = sum(inference_times) / len(inference_times)

print("=" * 55)
print("  HerCycle AI — PRELIMINARY METRICS REPORT")
print("  Food Classifier (Pretrained GoogLeNet Baseline)")
print("=" * 55)
print()
print("  PERFORMANCE METRICS")
print(f"  {'Metric':<20} {'Value':<10} {'Target'}")
print(f"  {'-'*45}")
print(f"  {'Accuracy':<20} {accuracy*100:.1f}%      > 80% (post fine-tune)")
print(f"  {'Precision':<20} {precision*100:.1f}%      > 75% (post fine-tune)")
print(f"  {'Recall':<20} {recall*100:.1f}%      > 75% (post fine-tune)")
print(f"  {'F1 Score':<20} {f1*100:.1f}%      > 75% (post fine-tune)")
print()
print("  SPEED METRICS")
print(f"  {'Avg Inference Time':<20} {avg_speed:.1f}ms    < 100ms (on Jetson)")
print(f"  {'Device':<20} CPU (laptop baseline)")
print(f"  {'Jetson Target':<20} < 100ms with TensorRT")
print()
print("  CONFUSION MATRIX")
print(f"  {'':>20} Predicted")
print(f"  {'':>20} Friendly  Unfriendly")
print(f"  Actual Friendly  :    {cm[1][1]}         {cm[1][0]}")
print(f"  Actual Unfriendly:    {cm[0][1]}         {cm[0][0]}")
print()
print(f"  TP={cm[1][1]}  FN={cm[1][0]}  FP={cm[0][1]}  TN={cm[0][0]}")
print()
print("  NOTES")
print("  - Baseline using pretrained ImageNet weights only")
print("  - 3 misclassifications due to ImageNet label mismatch")
print("  - Fine-tuning on Food-101 expected to exceed 80%")
print("=" * 55)

  HerCycle AI — PRELIMINARY METRICS REPORT
  Food Classifier (Pretrained GoogLeNet Baseline)

  PERFORMANCE METRICS
  Metric               Value      Target
  ---------------------------------------------
  Accuracy             70.0%      > 80% (post fine-tune)
  Precision            75.0%      > 75% (post fine-tune)
  Recall               60.0%      > 75% (post fine-tune)
  F1 Score             66.7%      > 75% (post fine-tune)

  SPEED METRICS
  Avg Inference Time   244.1ms    < 100ms (on Jetson)
  Device               CPU (laptop baseline)
  Jetson Target        < 100ms with TensorRT

  CONFUSION MATRIX
                       Predicted
                       Friendly  Unfriendly
  Actual Friendly  :    3         2
  Actual Unfriendly:    1         4

  TP=3  FN=2  FP=1  TN=4

  NOTES
  - Baseline using pretrained ImageNet weights only
  - 3 misclassifications due to ImageNet label mismatch
  - Fine-tuning on Food-101 expected to exceed 80%
